## 第一节 加载公开词向量

In [ ]:
from gensim.models import KeyedVectors

model_path = "./data/sgns.weibo.word.bz2"
model = KeyedVectors.load_word2vec_format(model_path)


In [ ]:
print(model['地铁'])

In [ ]:
similarity = model.similarity('地铁','公交')
print('地铁 vs 公交 相似度：',similarity)

In [ ]:
# 拿到维数
model.vector_size

In [ ]:
# 词数
len(model.index_to_key)

In [ ]:
# 最相似，支持计算
model.most_similar(positive=['男人','女孩'],negative=['男孩'],topn=5)

## 第二节 训练自己的词向量

In [ ]:
import pandas as pd
import jieba
from gensim.models import Word2Vec

In [ ]:
df = pd.read_csv("./data/online_shopping_10_cats.csv",encoding="utf-8").dropna()
df.head()

In [ ]:
df[df['review'].isna()]

In [ ]:
sentences = [[token for token in jieba.lcut(sentence) if token.strip() != ''] for sentence in df['review']]


In [ ]:
model = Word2Vec(
    sentences,          # 已分词的句子序列
    vector_size=100,    # 词向量维度
    window=5,           # 上下文窗口大小
    min_count=2,        # 最小词频,低于该词频将直接被忽略
    sg=1,               # 1 Skip-Gram 0 CBOW
    workers=5           # 并行训练线程数
)

In [ ]:
model.wv['地铁']

In [ ]:
model.wv.vector_size

In [ ]:
model.wv.save_word2vec_format('./data/word2vec.txt') # 将词表持久化

# 随机初始化
模型训练开始时,嵌入向量是随机生成的,模型会通过反向传播逐步学习每个词的表示.
# 使用预训练词向量初始化
加载训练好的词向量到嵌入层中作为初始参数,这样可以为模型注入丰富的语言知识,尤其在低资源任务中优势明显.并且,加载预训练词向量后,可选择是否让嵌入层继续参与训练.


# 第三节 词向量的具体应用


In [ ]:
from torch import nn
import torch
import jieba

In [ ]:
# 1. 加载词向量
wv = KeyedVectors.load_word2vec_format('./data/word2vec.txt')

# 1.1 处理OOV Out Of Vocabulary
unk_token = '<unk>'
index2word = [unk_token]+wv.index_to_key
word2index = {word:index for index,word in enumerate(index2word)}


In [ ]:
# 2. 准备词向量矩阵
#num_embeddings = len(wv.index_to_key)
num_embeddings = len(index2word)
embedding_dim = wv.vector_size
embedding_matrix = torch.randn(num_embeddings,embedding_dim)


# for (idx,word) in enumerate(wv.index_to_key):
#     embedding_matrix[idx] = torch.tensor(wv[word])
for (idx,word) in enumerate(index2word):
    if word in wv:
        embedding_matrix[idx] = torch.tensor(wv[word])




In [ ]:
# 3. 创建Embedding
embbeding = nn.Embedding.from_pretrained(embedding_matrix)

In [ ]:
# 4. 测试
text = "我喜欢乘坐宇宙飞船"
tokens = jieba.lcut(text)
# 将分词转成词向量
# input_ids = [wv.key_to_index[token] for token in tokens]
input_ids = [wv.key_to_index.get(token,word2index[unk_token]) for token in tokens]
print(input_ids)
input_tensor = torch.tensor(input_ids)
embbeding(input_tensor).shape